In [9]:
import pandas as pd
from pathlib import Path
import sqlite3

def carica_e_pulisci_cartella(nome_cartella, nome_campionato):
    # Path(nome_cartella) cerca la sottocartella partendo da dove si trova il Notebook
    percorso = Path(nome_cartella)
    
    if not percorso.exists():
        raise FileNotFoundError(f"⚠️ Cartella '{nome_cartella}' non trovata! Assicurati che il Notebook sia dentro Progetto_Football.")
    
    # Trova tutti i file .csv (sia minuscolo che maiuscolo)
    tutti_i_file = list(percorso.glob("*.csv")) + list(percorso.glob("*.CSV"))
    print(f"📂 Trovati {len(tutti_i_file)} file CSV nella cartella '{nome_cartella}'")
    
    lista_df = []
    
    colonne_target = [
        'Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR',
        'HS', 'AS', 'HST', 'AST', 'HC', 'AC', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR',
        'B365H', 'B365D', 'B365A'
    ]
    
    for file in tutti_i_file:
        try:
            df_temp = pd.read_csv(file, encoding='latin1', on_bad_lines='skip')
        except Exception as e:
            print(f"❌ Errore nel file {file.name}: {e}")
            continue
            
        # Scarta file che non hanno le colonne base delle partite
        if not {'Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG'}.issubset(df_temp.columns):
            print(f"⚠️ Salto {file.name}: mancano le colonne principali.")
            continue
            
        # Rimuove le righe vuote di fine file
        df_temp = df_temp.dropna(subset=['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG'])
        
        # Filtra solo le colonne target presenti
        colonne_presenti = [c for c in colonne_target if c in df_temp.columns]
        df_temp = df_temp[colonne_presenti].copy()
        
        df_temp['League'] = nome_campionato
        df_temp['SourceFile'] = file.name
        
        lista_df.append(df_temp)
        
    df_unito = pd.concat(lista_df, ignore_index=True)
    
    # Converte le date (formato europeo Giorno/Mese/Anno)
    df_unito['Date'] = pd.to_datetime(df_unito['Date'], dayfirst=True, format='mixed')
    
    # Converte i goal in numeri interi
    df_unito['FTHG'] = df_unito['FTHG'].astype(int)
    df_unito['FTAG'] = df_unito['FTAG'].astype(int)
    
    return df_unito

# --- ESECUZIONE ---
# Qui passiamo esattamente i nomi delle tue due cartelle: "SerieA" e "SerieB"
df_a = carica_e_pulisci_cartella("SerieA", "Serie_A")
df_b = carica_e_pulisci_cartella("SerieB", "Serie_B")

# Uniamo i due campionati e ordiniamo per data
df_totale = pd.concat([df_a, df_b], ignore_index=True)
df_totale = df_totale.sort_values('Date').reset_index(drop=True)

# Rimuoviamo eventuali partite duplicate (se per errore hai scaricato due volte la stessa stagione)
df_totale = df_totale.drop_duplicates(subset=['Date', 'HomeTeam', 'AwayTeam'], keep='first').reset_index(drop=True)

print(f"\n✅ Caricamento completato! Totale partite uniche: {len(df_totale)}")
print(f"📅 Intervallo date: dal {df_totale['Date'].min().date()} al {df_totale['Date'].max().date()}")
display(df_totale.head())

# Rimuove eventuali spazi invisibili prima o dopo i nomi delle squadre (es. 'Piacenza ')
df_totale['HomeTeam'] = df_totale['HomeTeam'].str.strip()
df_totale['AwayTeam'] = df_totale['AwayTeam'].str.strip()

📂 Trovati 60 file CSV nella cartella 'SerieA'
📂 Trovati 60 file CSV nella cartella 'SerieB'

✅ Caricamento completato! Totale partite uniche: 21929
📅 Intervallo date: dal 1997-08-31 al 2026-10-28


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HST,AST,...,AF,HY,AY,HR,AR,B365H,B365D,B365A,League,SourceFile
0,1997-08-31,Salernitana,Verona,2,0,H,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Serie_B,I2 (29).csv
1,1997-08-31,Atalanta,Bologna,4,2,H,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Serie_A,I1 (29).csv
2,1997-08-31,Bari,Parma,0,2,A,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Serie_A,I1 (29).csv
3,1997-08-31,Empoli,Roma,1,3,A,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Serie_A,I1 (29).csv
4,1997-08-31,Inter,Brescia,2,1,H,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Serie_A,I1 (29).csv


In [10]:
import numpy as np

# 1. Normalizzazione rapida di eventuali nomi duplicati nei CSV storici
mappa_nomi = {
    'Internazionale': 'Inter',
    'Hellas Verona': 'Verona',
    'AC Milan': 'Milan',
    'Spal': 'SPAL'
}
df_totale['HomeTeam'] = df_totale['HomeTeam'].replace(mappa_nomi)
df_totale['AwayTeam'] = df_totale['AwayTeam'].replace(mappa_nomi)

# 2. Funzione per calcolare l'Elo Rating (Forza storica dinamica della squadra)
def aggiungi_elo_rating(df, k_factor=20, home_advantage=65):
    elo_dict = {}
    home_elos, away_elos = [], []
    
    for idx, row in df.iterrows():
        home, away, league = row['HomeTeam'], row['AwayTeam'], row['League']
        
        # Chi parte in Serie A inizia da 1500, chi parte in Serie B da 1350
        if home not in elo_dict:
            elo_dict[home] = 1500.0 if league == 'Serie_A' else 1350.0
        if away not in elo_dict:
            elo_dict[away] = 1500.0 if league == 'Serie_A' else 1350.0
            
        elo_h, elo_a = elo_dict[home], elo_dict[away]
        
        # Salviamo l'Elo PRIMA del match
        home_elos.append(elo_h)
        away_elos.append(elo_a)
        
        # Probabilità attesa di vittoria in base alla differenza Elo
        exp_h = 1.0 / (1.0 + 10.0 ** ((elo_a - (elo_h + home_advantage)) / 400.0))
        exp_a = 1.0 - exp_h
        
        # Risultato reale (1 = Vittoria, 0.5 = Pareggio, 0 = Sconfitta)
        if row['FTHG'] > row['FTAG']:
            score_h, score_a = 1.0, 0.0
        elif row['FTHG'] < row['FTAG']:
            score_h, score_a = 0.0, 1.0
        else:
            score_h, score_a = 0.5, 0.5
            
        # Moltiplicatore per lo scarto di goal (premia le goleade)
        goal_diff = abs(row['FTHG'] - row['FTAG'])
        mult = 1.0 if goal_diff <= 1 else (1.5 if goal_diff == 2 else (11.0 + goal_diff) / 8.0)
        
        # Aggiorniamo l'Elo per la partita successiva
        elo_dict[home] = elo_h + k_factor * mult * (score_h - exp_h)
        elo_dict[away] = elo_a + k_factor * mult * (score_a - exp_a)
        
    df['Home_Elo'] = home_elos
    df['Away_Elo'] = away_elos
    df['Elo_Diff'] = df['Home_Elo'] - df['Away_Elo']
    return df

df_totale = aggiungi_elo_rating(df_totale)

# 3. Calcolo delle Rolling Features (Stato di forma nelle ultime 5 partite)
# Assegniamo un ID univoco a ogni partita per ricomporre il dataset facilmente
df_totale['Match_ID'] = df_totale.index

# Creiamo una vista "per squadra" (ogni partita genera 2 righe: una vista dalla squadra di casa, una dall'ospite)
home_view = pd.DataFrame({
    'Match_ID': df_totale['Match_ID'],
    'Date': df_totale['Date'],
    'Team': df_totale['HomeTeam'],
    'Is_Home': 1,
    'Goals_For': df_totale['FTHG'],
    'Goals_Against': df_totale['FTAG'],
    'Shots_Target_For': df_totale['HST'] if 'HST' in df_totale.columns else np.nan,
    'Shots_Target_Against': df_totale['AST'] if 'AST' in df_totale.columns else np.nan,
    'Points': np.where(df_totale['FTHG'] > df_totale['FTAG'], 3, np.where(df_totale['FTHG'] == df_totale['FTAG'], 1, 0))
})

away_view = pd.DataFrame({
    'Match_ID': df_totale['Match_ID'],
    'Date': df_totale['Date'],
    'Team': df_totale['AwayTeam'],
    'Is_Home': 0,
    'Goals_For': df_totale['FTAG'],
    'Goals_Against': df_totale['FTHG'],
    'Shots_Target_For': df_totale['AST'] if 'AST' in df_totale.columns else np.nan,
    'Shots_Target_Against': df_totale['HST'] if 'HST' in df_totale.columns else np.nan,
    'Points': np.where(df_totale['FTAG'] > df_totale['FTHG'], 3, np.where(df_totale['FTHG'] == df_totale['FTAG'], 1, 0))
})

# Uniamo e ordiniamo cronologicamente per squadra
team_log = pd.concat([home_view, away_view]).sort_values(['Team', 'Date', 'Match_ID']).reset_index(drop=True)

# Calcoliamo le medie delle ULTIME 5 PARTITE usando .shift(1) per escludere la partita corrente!
cols_to_roll = ['Goals_For', 'Goals_Against', 'Shots_Target_For', 'Shots_Target_Against', 'Points']
for col in cols_to_roll:
    team_log[f'Roll5_{col}'] = (
        team_log.groupby('Team')[col]
        .transform(lambda x: x.shift(1).rolling(window=5, min_periods=3).mean())
    )

# 4. Riportiamo le medie calcolate dentro df_totale
roll_cols = [f'Roll5_{c}' for c in cols_to_roll]

home_features = team_log[team_log['Is_Home'] == 1][['Match_ID'] + roll_cols].rename(
    columns={c: f'Home_{c}' for c in roll_cols}
)
away_features = team_log[team_log['Is_Home'] == 0][['Match_ID'] + roll_cols].rename(
    columns={c: f'Away_{c}' for c in roll_cols}
)

df_features = df_totale.merge(home_features, on='Match_ID').merge(away_features, on='Match_ID')

# Rimuoviamo le prime partite in cui le squadre non hanno ancora almeno 3 gare di storico
df_ml = df_features.dropna(subset=['Home_Roll5_Goals_For', 'Away_Roll5_Goals_For']).reset_index(drop=True)

print(f"✅ Dataset pronto per il Machine Learning! Partite utilizzabili: {len(df_ml)}")
display(df_ml[['Date', 'League', 'HomeTeam', 'AwayTeam', 'Home_Elo', 'Away_Elo', 'Home_Roll5_Goals_For', 'Away_Roll5_Goals_For']].tail())

✅ Dataset pronto per il Machine Learning! Partite utilizzabili: 21724


,Date,League,HomeTeam,AwayTeam,Home_Elo,Away_Elo,Home_Roll5_Goals_For,Away_Roll5_Goals_For
21719,2026-10-28,Serie_A,Monza,Napoli,1500.229420,1770.655060,2.2,1.6
21720,2026-10-28,Serie_A,Genoa,Juventus,1550.110767,1752.659369,1.0,1.4
21721,2026-10-28,Serie_A,Venezia,Inter,1506.377016,1933.291897,0.8,3.0
21722,2026-10-28,Serie_A,Milan,Bologna,1766.436174,1646.857899,2.2,1.2
21723,2026-10-28,Serie_B,Juve Stabia,Verona,1365.291799,1404.758516,1.4,1.6


In [11]:
import numpy as np
import pandas as pd
from scipy.stats import poisson
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import PoissonRegressor
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, log_loss, mean_absolute_error

# 1. Prepariamo le Feature (X) e i Target (y)
df_ml['Is_Serie_A'] = (df_ml['League'] == 'Serie_A').astype(int)

features = [
    'Is_Serie_A', 'Home_Elo', 'Away_Elo', 'Elo_Diff',
    'Home_Roll5_Goals_For', 'Home_Roll5_Goals_Against', 'Home_Roll5_Points',
    'Away_Roll5_Goals_For', 'Away_Roll5_Goals_Against', 'Away_Roll5_Points',
    'Home_Roll5_Shots_Target_For', 'Home_Roll5_Shots_Target_Against',
    'Away_Roll5_Shots_Target_For', 'Away_Roll5_Shots_Target_Against'
]

X = df_ml[features].copy()
X = X.fillna(X.median())

y_home_goals = df_ml['FTHG']
y_away_goals = df_ml['FTAG']
y_1x2 = df_ml['FTR'].map({'A': 0, 'D': 1, 'H': 2})

# 2. Split Temporale (85% Training, 15% Test)
split_idx = int(len(df_ml) * 0.85)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
yh_train, yh_test = y_home_goals.iloc[:split_idx], y_home_goals.iloc[split_idx:]
ya_train, ya_test = y_away_goals.iloc[:split_idx], y_away_goals.iloc[split_idx:]
y1x2_train, y1x2_test = y_1x2.iloc[:split_idx], y_1x2.iloc[split_idx:]
df_test_info = df_ml.iloc[split_idx:].copy()

# 3. ADDESTRAMENTO MODELLO 1: StandardScaler + PoissonRegressor (Evita l'overflow di matmul!)
poisson_home = make_pipeline(StandardScaler(), PoissonRegressor(alpha=1.0, max_iter=500))
poisson_away = make_pipeline(StandardScaler(), PoissonRegressor(alpha=1.0, max_iter=500))

poisson_home.fit(X_train, yh_train)
poisson_away.fit(X_train, ya_train)

df_test_info['Exp_Home_Goals'] = np.round(poisson_home.predict(X_test), 2)
df_test_info['Exp_Away_Goals'] = np.round(poisson_away.predict(X_test), 2)
df_test_info['Exp_Total_Goals'] = df_test_info['Exp_Home_Goals'] + df_test_info['Exp_Away_Goals']

# 4. ADDESTRAMENTO MODELLO 2: Gradient Boosting per l'Esito 1X2
clf_1x2 = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, max_depth=4, random_state=42)
clf_1x2.fit(X_train, y1x2_train)

probs_1x2 = clf_1x2.predict_proba(X_test)
preds_1x2 = clf_1x2.predict(X_test)

df_test_info['Prob_1 (%)'] = np.round(probs_1x2[:, 2] * 100, 1)
df_test_info['Prob_X (%)'] = np.round(probs_1x2[:, 1] * 100, 1)
df_test_info['Prob_2 (%)'] = np.round(probs_1x2[:, 0] * 100, 1)

mappa_inversa = {2: '1', 1: 'X', 0: '2'}
df_test_info['Pred_Esito'] = [mappa_inversa[p] for p in preds_1x2]
df_test_info['Reale_Esito'] = df_test_info['FTR'].map({'H': '1', 'D': 'X', 'A': '2'})

# 5. Stampa metriche di valutazione
acc = accuracy_score(y1x2_test, preds_1x2)
lloss = log_loss(y1x2_test, probs_1x2)
mae_tot = mean_absolute_error(yh_test + ya_test, df_test_info['Exp_Total_Goals'])

print(f"📊 RISULTATI SUL TEST SET ({len(X_test)} partite recenti):")
print(f"✔️ Accuratezza Esito 1X2: {acc*100:.2f}%")
print(f"📉 Log-Loss 1X2: {lloss:.4f}")
print(f"⚽ Errore Medio Assoluto sui Goal Totali: {mae_tot:.2f} goal")

colonne_output = [
    'Date', 'League', 'HomeTeam', 'AwayTeam', 
    'FTHG', 'FTAG', 'Exp_Home_Goals', 'Exp_Away_Goals',
    'Prob_1 (%)', 'Prob_X (%)', 'Prob_2 (%)', 'Pred_Esito', 'Reale_Esito'
]
display(df_test_info[colonne_output].tail(10))

📊 RISULTATI SUL TEST SET (3259 partite recenti):
✔️ Accuratezza Esito 1X2: 46.92%
📉 Log-Loss 1X2: 1.0274
⚽ Errore Medio Assoluto sui Goal Totali: 1.26 goal


,Date,League,HomeTeam,AwayTeam,FTHG,FTAG,Exp_Home_Goals,Exp_Away_Goals,Prob_1 (%),Prob_X (%),Prob_2 (%),Pred_Esito,Reale_Esito
21714,2026-10-27,Serie_A,Roma,Cagliari,4,2,2.07,0.83,75.9,14.1,10.0,1,1
21715,2026-10-27,Serie_A,Torino,Como,1,3,1.26,1.37,25.2,21.3,53.4,2,2
21716,2026-10-28,Serie_B,Pisa,Palermo,3,3,1.31,1.27,35.3,32.6,32.2,1,X
21717,2026-10-28,Serie_A,Parma,Udinese,0,0,1.54,1.10,41.7,33.5,24.8,1,X
21718,2026-10-28,Serie_B,Carrarese,Arezzo,2,1,1.33,0.98,38.0,34.6,27.4,1,1
21719,2026-10-28,Serie_A,Monza,Napoli,1,4,1.16,1.67,12.5,21.2,66.2,2,2
21720,2026-10-28,Serie_A,Genoa,Juventus,2,0,1.19,1.37,25.8,20.8,53.4,2,1
21721,2026-10-28,Serie_A,Venezia,Inter,0,2,0.88,2.22,13.0,20.4,66.6,2,2
21722,2026-10-28,Serie_A,Milan,Bologna,2,4,1.83,0.98,58.5,26.9,14.6,1,2
21723,2026-10-28,Serie_B,Juve Stabia,Verona,0,1,1.43,1.01,43.1,32.8,24.2,1,2


In [12]:
import numpy as np
import pandas as pd
from scipy.stats import poisson
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import PoissonRegressor
from sklearn.ensemble import HistGradientBoostingClassifier

# 1. Ri-addestriamo i modelli sul 100% dei dati USANDO LA PIPELINE CON STANDARDSCALER
# In questo modo l'Elo (es. 1550) viene normalizzato e sparisce il RuntimeWarning!
poisson_home_full = make_pipeline(
    StandardScaler(), 
    PoissonRegressor(alpha=1.0, max_iter=500)
).fit(X, y_home_goals)

poisson_away_full = make_pipeline(
    StandardScaler(), 
    PoissonRegressor(alpha=1.0, max_iter=500)
).fit(X, y_away_goals)

clf_1x2_full = HistGradientBoostingClassifier(
    max_iter=150, learning_rate=0.05, max_depth=4, random_state=42
).fit(X, y_1x2)

# 2. Funzione per estrarre lo stato attuale (Elo + ultime 5 partite) di una singola squadra
def get_stato_attuale_squadra(nome_squadra):
    nome_squadra = nome_squadra.strip() # Rimuove eventuali spazi extra
    partite_squadra = team_log[team_log['Team'] == nome_squadra].sort_values('Date')
    
    if len(partite_squadra) < 5:
        raise ValueError(f"⚠️ Squadra '{nome_squadra}' non trovata o con meno di 5 partite. Controlla il nome esatto!")
        
    ultime_5 = partite_squadra.tail(5)
    
    # Calcoliamo le medie sulle ultime 5 partite giocate
    roll_gf = ultime_5['Goals_For'].mean()
    roll_ga = ultime_5['Goals_Against'].mean()
    roll_pts = ultime_5['Points'].mean()
    roll_stf = ultime_5['Shots_Target_For'].mean()
    roll_sta = ultime_5['Shots_Target_Against'].mean()
    
    # Recuperiamo l'ultimo Elo disponibile
    ultima_partita_id = ultime_5.iloc[-1]['Match_ID']
    riga_match = df_totale.loc[ultima_partita_id]
    
    if riga_match['HomeTeam'].strip() == nome_squadra:
        elo_attuale = riga_match['Home_Elo']
    else:
        elo_attuale = riga_match['Away_Elo']
        
    return {
        'Elo': elo_attuale,
        'Roll5_Goals_For': roll_gf,
        'Roll5_Goals_Against': roll_ga,
        'Roll5_Points': roll_pts,
        'Roll5_Shots_Target_For': roll_stf if not np.isnan(roll_stf) else X['Home_Roll5_Shots_Target_For'].median(),
        'Roll5_Shots_Target_Against': roll_sta if not np.isnan(roll_sta) else X['Home_Roll5_Shots_Target_Against'].median()
    }

# 3. Funzione principale per predire qualsiasi partita futura
def predici_partita(squadra_casa, squadra_ospite, campionato="Serie_A"):
    stato_h = get_stato_attuale_squadra(squadra_casa)
    stato_a = get_stato_attuale_squadra(squadra_ospite)
    
    is_serie_a = 1 if campionato == "Serie_A" else 0
    elo_diff = stato_h['Elo'] - stato_a['Elo']
    
    # Costruiamo il DataFrame a 1 riga con lo stesso identico ordine di 'features'
    x_nuova = pd.DataFrame([{
        'Is_Serie_A': is_serie_a,
        'Home_Elo': stato_h['Elo'],
        'Away_Elo': stato_a['Elo'],
        'Elo_Diff': elo_diff,
        'Home_Roll5_Goals_For': stato_h['Roll5_Goals_For'],
        'Home_Roll5_Goals_Against': stato_h['Roll5_Goals_Against'],
        'Home_Roll5_Points': stato_h['Roll5_Points'],
        'Away_Roll5_Goals_For': stato_a['Roll5_Goals_For'],
        'Away_Roll5_Goals_Against': stato_a['Roll5_Goals_Against'],
        'Away_Roll5_Points': stato_a['Roll5_Points'],
        'Home_Roll5_Shots_Target_For': stato_h['Roll5_Shots_Target_For'],
        'Home_Roll5_Shots_Target_Against': stato_h['Roll5_Shots_Target_Against'],
        'Away_Roll5_Shots_Target_For': stato_a['Roll5_Shots_Target_For'],
        'Away_Roll5_Shots_Target_Against': stato_a['Roll5_Shots_Target_Against']
    }])
    
    # A) Predizione Goal Attesi (La pipeline scala automaticamente x_nuova prima di predire!)
    lam_h = poisson_home_full.predict(x_nuova)[0]
    lam_a = poisson_away_full.predict(x_nuova)[0]
    
    # B) Predizione Esito 1X2 tramite Gradient Boosting
    probs_gb = clf_1x2_full.predict_proba(x_nuova)[0] # [Prob_2, Prob_X, Prob_1]
    p1_gb, px_gb, p2_gb = probs_gb[2] * 100, probs_gb[1] * 100, probs_gb[0] * 100
    
    # C) Matrice di Poisson per Risultati Esatti (fino a 6 goal per squadra), Over/Under 2.5 e Goal/NoGoal
    max_goals = 7
    prob_matrix = np.outer(poisson.pmf(range(max_goals), lam_h), poisson.pmf(range(max_goals), lam_a))
    
    prob_over_25 = sum(prob_matrix[i, j] for i in range(max_goals) for j in range(max_goals) if (i + j) > 2) * 100
    prob_under_25 = 100.0 - prob_over_25
    prob_btts = sum(prob_matrix[i, j] for i in range(1, max_goals) for j in range(1, max_goals)) * 100
    
    risultati_esatti = []
    for i in range(max_goals):
        for j in range(max_goals):
            risultati_esatti.append((f"{i}-{j}", prob_matrix[i, j] * 100))
    risultati_esatti.sort(key=lambda x: x[1], reverse=True)
    top3_risultati = risultati_esatti[:3]
    
    # --- STAMPA DEL REPORT ---
    print("=" * 60)
    print(f"🏟️  MATCH: {squadra_casa} vs {squadra_ospite} ({campionato})")
    print("=" * 60)
    print(f"⚡ Elo Rating:      {squadra_casa} ({stato_h['Elo']:.0f}) | {squadra_ospite} ({stato_a['Elo']:.0f}) [Diff: {elo_diff:+.0f}]")
    print(f"📈 Forma (ult. 5):  {squadra_casa} ({stato_h['Roll5_Points']*5:.0f} pt, {stato_h['Roll5_Goals_For']:.1f} GF/g) | "
          f"{squadra_ospite} ({stato_a['Roll5_Points']*5:.0f} pt, {stato_a['Roll5_Goals_For']:.1f} GF/g)")
    print("-" * 60)
    print(f"⚽ GOAL ATTESI (xG Modello):  {squadra_casa}: {lam_h:.2f}  |  {squadra_ospite}: {lam_a:.2f}  (Totale: {lam_h+lam_a:.2f})")
    print("-" * 60)
    print(f"🎯 PROBABILITÀ 1X2 (Gradient Boosting):")
    print(f"   [1] Vittoria {squadra_casa:<12}: {p1_gb:5.1f}%  (Quota equa: {100/p1_gb:.2f})")
    print(f"   [X] Pareggio             : {px_gb:5.1f}%  (Quota equa: {100/px_gb:.2f})")
    print(f"   [2] Vittoria {squadra_ospite:<12}: {p2_gb:5.1f}%  (Quota equa: {100/p2_gb:.2f})")
    print("-" * 60)
    print(f"📊 MERCATI GOAL (Distribuzione di Poisson):")
    print(f"   • Over 2.5 Goal : {prob_over_25:5.1f}%  |  Under 2.5 Goal: {prob_under_25:5.1f}%")
    print(f"   • Goal (BTTS Sì): {prob_btts:5.1f}%  |  No Goal       : {100-prob_btts:5.1f}%")
    print("-" * 60)
    print(f"🔮 TOP 3 RISULTATI ESATTI PIÙ PROBABILI:")
    for idx, (ris, prob) in enumerate(top3_risultati, 1):
        print(f"   {idx}°)  {ris}  ({prob:.1f}%)")
    print("=" * 60)

# --- PROVA SUBITO UNA PARTITA DI SERIE A E UNA DI SERIE B ---
predici_partita("Fiorentina", "Atalanta", campionato="Serie_A")
#predici_partita("Pisa", "Spezia", campionato="Serie_B")

#Per vedere la lista esatta di come sono scritte tutte le squadre di Serie A e Serie B nei tuoi file, ti basta #eseguire in una cella:

#print(sorted(df_totale['HomeTeam'].unique()))

🏟️  MATCH: Fiorentina vs Atalanta (Serie_A)
⚡ Elo Rating:      Fiorentina (1596) | Atalanta (1712) [Diff: -116]
📈 Forma (ult. 5):  Fiorentina (4 pt, 1.6 GF/g) | Atalanta (4 pt, 2.0 GF/g)
------------------------------------------------------------
⚽ GOAL ATTESI (xG Modello):  Fiorentina: 1.39  |  Atalanta: 1.37  (Totale: 2.76)
------------------------------------------------------------
🎯 PROBABILITÀ 1X2 (Gradient Boosting):
   [1] Vittoria Fiorentina  :  26.8%  (Quota equa: 3.74)
   [X] Pareggio             :  26.4%  (Quota equa: 3.79)
   [2] Vittoria Atalanta    :  46.8%  (Quota equa: 2.14)
------------------------------------------------------------
📊 MERCATI GOAL (Distribuzione di Poisson):
   • Over 2.5 Goal :  52.0%  |  Under 2.5 Goal:  48.0%
   • Goal (BTTS Sì):  56.0%  |  No Goal       :  44.0%
------------------------------------------------------------
🔮 TOP 3 RISULTATI ESATTI PIÙ PROBABILI:
   1°)  1-1  (12.0%)
   2°)  1-0  (8.8%)
   3°)  0-1  (8.6%)
